In [4]:
from pathlib import Path
import pandas as pd
import shutil

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

source_dir = Path("/Users/sm6511/Downloads/pafresh2")

destination_dir = Path(
    "/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/Study6.0/raw"
)

destination_dir.mkdir(parents=True, exist_ok=True)

# Inclusion columns
completion_column = "debrief_survey.block_1/debriefComplete"
attention_column = "answer_3_right.numClicks"


# ------------------------------------------------------------
# Loop through raw CSV files
# ------------------------------------------------------------

copied_files = []
skipped_files = []
error_files = []

for file_path in source_dir.glob("*.csv"):

    try:
        df = pd.read_csv(file_path)

        # Check required columns exist
        missing_columns = [
            col for col in [completion_column, attention_column]
            if col not in df.columns
        ]

        if missing_columns:
            print(
                f"Missing column(s) {missing_columns}: "
                f"{file_path.name}"
            )
            skipped_files.append(file_path.name)
            continue

        # ----------------------------------------------------
        # Criterion 1: Experiment completed
        # ----------------------------------------------------

        completed = (
            df[completion_column]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("complete")
            .any()
        )

        # ----------------------------------------------------
        # Criterion 2: Attention check passed
        # ----------------------------------------------------

        attention_passed = (
            pd.to_numeric(
                df[attention_column],
                errors="coerce"
            )
            .eq(1)
            .any()
        )

        # ----------------------------------------------------
        # Must satisfy BOTH criteria
        # ----------------------------------------------------

        if completed and attention_passed:

            destination_path = destination_dir / file_path.name

            shutil.copy2(
                file_path,
                destination_path
            )

            copied_files.append(file_path.name)

            print(f"COPIED: {file_path.name}")

        else:

            skipped_files.append(file_path.name)

            # Helpful information about why it was excluded
            reasons = []

            if not completed:
                reasons.append("incomplete")

            if not attention_passed:
                reasons.append("failed attention check")

            print(
                f"SKIPPED: {file_path.name} "
                f"({', '.join(reasons)})"
            )

    except Exception as e:

        error_files.append(file_path.name)

        print(f"ERROR reading {file_path.name}: {e}")


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n------------------------------")
print("Finished")
print("------------------------------")
print(f"Copied:  {len(copied_files)}")
print(f"Skipped: {len(skipped_files)}")
print(f"Errors:  {len(error_files)}")

COPIED: 310688_pa6fresh_2026-08-13_13h06.16.204.csv
COPIED: 356916_pa6fresh_2026-08-13_15h47.31.409.csv
COPIED: 788793_pa6fresh_2026-08-13_16h12.53.759.csv
COPIED: 615902_pa6fresh_2026-08-13_14h37.32.350.csv
COPIED: 817664_pa6fresh_2026-08-13_14h36.12.432.csv
COPIED: 031362_pa6fresh_2026-08-13_14h45.55.264.csv
COPIED: 662588_pa6fresh_2026-08-13_16h28.56.786.csv
COPIED: 859284_pa6fresh_2026-08-13_17h20.12.974.csv
COPIED: 063752_pa6fresh_2026-08-13_16h47.55.440.csv
COPIED: 340858_pa6fresh_2026-08-13_13h42.24.314.csv
COPIED: 778390_pa6fresh_2026-08-13_13h06.18.563.csv
COPIED: 218395_pa6fresh_2026-08-13_14h38.55.770.csv
COPIED: 126419_pa6fresh_2026-08-13_15h39.05.489.csv
Missing column(s) ['debrief_survey.block_1/debriefComplete', 'answer_3_right.numClicks']: 518268_pa6fresh_2026-08-13_17h47.35.579.csv
ERROR reading 732785_pa6fresh_2026-08-13_14h54.19.565.csv: No columns to parse from file
ERROR reading 444060_pa6fresh_2026-08-13_17h47.02.957.csv: No columns to parse from file
COPIED: 6030